# [1] Imports & Motor-CAD 연결


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [1] Imports & Motor-CAD 연결
# ─────────────────────────────────────────────────────────────────────────────
import pathlib
import sys
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat

# Repo root on path
repo_root = pathlib.Path.cwd().resolve()
while not ((repo_root / "tools").exists() or (repo_root / "tool").exists()) and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 패키지 reload (의존순서: model → plot → parse → facade)
import tools.motorCAD.pyMCAD.magnetic_model as _mm
import tools.motorCAD.pyMCAD.magnetic_plot as _mp
import tools.motorCAD.pyMCAD.magnetic_parse as _mparse
import tools.motorCAD.pyMCAD.magnetic as _mag
importlib.reload(_mm)
importlib.reload(_mp)
importlib.reload(_mparse)
importlib.reload(_mag)

from tools.motorCAD.pyMCAD import (
    get_magnetic_timeseries_from_file,
    mcad_default_export_dir,
    find_latest_mes,
    list_mes_files,
)
from tools.motorCAD.pyMCAD.magnetic_model import MagElement

import ansys.motorcad.core as pymotorcad

# Motor-CAD 연결
mcad = pymotorcad.MotorCAD(open_new_instance=False)
# refMotFilePath=r"D:\KangDH\Thesis\e10\refModel\e10Turn6V261.mot"
HalfSCMotFilePath=r"D:\KangDH\Thesis\e10\SLFEA_Half\e10Turn6V261SLFEA_Half.mot"
# SCMotFilePath=r"D:\KangDH\Thesis\e10\SLFEA\e10Turn6V261SLFEA.mot"

# mcad.load_from_file(refMotFilePath)
print("Motor-CAD connected")
# print(f"  MOT file: {mcad.get_variable('CurrentMotFilePath_MotorLAB')}")

Motor-CAD connected


# [2] e10 모터 파라미터 설정

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# [2] e10 모터 파라미터 설정
# ─────────────────────────────────────────────────────────────────────────────

# --- Conductor geometry (hairpin, rectangular) ---
# COND_WIDTH_MM = 2.5        # tangential width b [mm] (확인 필요 → Motor-CAD에서)
# COND_HEIGHT_MM = 2.5       # radial height h [mm] (확인 필요 → Motor-CAD에서)
SIGMA_CU = 5.8e7           # Cu conductivity @ 20°C [S/m]
# ACTIVE_LENGTH_MM = 100.0   # axial stack length [mm] (확인 필요)

# Motor-CAD에서 실제 값 읽기
try:
    COND_WIDTH_MM = float(mcad.get_variable("Copper_Width"))  # 슬롯 폭 / 병렬 수
    COND_HEIGHT_MM = float(mcad.get_variable("Copper_Height"))
    ACTIVE_LENGTH_MM = float(mcad.get_variable("Stator_Lam_Length"))
    n_parallel = int(mcad.get_variable("ParallelPaths"))
    n_turns = int(mcad.get_variable("MagTurnsConductor"))
    print(f"  Conductor: {COND_WIDTH_MM:.2f} x {COND_HEIGHT_MM:.2f} mm")
    print(f"  Active length: {ACTIVE_LENGTH_MM:.1f} mm")
    print(f"  Parallel paths: {n_parallel}, Turns/conductor: {n_turns}")
except Exception as e:
    print(f"  [WARN] Motor-CAD variable read failed: {e}")
    print(f"  Using default values: {COND_WIDTH_MM} x {COND_HEIGHT_MM} mm, L={ACTIVE_LENGTH_MM} mm")

# Convert to SI
b_m = COND_WIDTH_MM * 1e-3     # conductor tangential width [m]
h_m = COND_HEIGHT_MM * 1e-3    # conductor radial height [m]
L_a = ACTIVE_LENGTH_MM * 1e-3  # active length [m]

# --- Operating conditions ---
POLE_PAIRS = 4                 # 8-pole motor
SPEED_LIST = [2000, 4000, 16000]  # RPM

# Electrical frequency per speed
def speed_to_fe(speed_rpm, pole_pairs=POLE_PAIRS):
    return pole_pairs * speed_rpm / 60.0

print(f"\n  Speed → f_e: {[(s, f'{speed_to_fe(s):.0f} Hz') for s in SPEED_LIST]}")

# --- Skin depth & ξ table ---
MU_0 = 4 * np.pi * 1e-7
print(f"\n{'Speed [RPM]':>12} {'f_e [Hz]':>10} {'δ [mm]':>10} {'ξ = h/δ':>10}")
print("-" * 50)
for spd in SPEED_LIST:
    fe = speed_to_fe(spd)
    delta = 1.0 / np.sqrt(np.pi * fe * MU_0 * SIGMA_CU)
    xi_val = h_m / delta
    print(f"{spd:>12} {fe:>10.0f} {delta*1e3:>10.2f} {xi_val:>10.3f}")

  Conductor: 5.57 x 2.53 mm
  Active length: 150.0 mm
  Parallel paths: 1, Turns/conductor: 1

  Speed → f_e: [(2000, '133 Hz'), (4000, '267 Hz'), (16000, '1067 Hz')]

 Speed [RPM]   f_e [Hz]     δ [mm]    ξ = h/δ
--------------------------------------------------
        2000        133       5.72      0.442
        4000        267       4.05      0.625
       16000       1067       2.02      1.250


# [3] FEA 설정 + B-field TXT Export (속도별)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [3] FEA 설정 + B-field TXT Export (속도별)
# ─────────────────────────────────────────────────────────────────────────────
# 
# 옵션 A: Motor-CAD를 여기서 직접 실행
# 옵션 B: 이미 실행된 결과의 .mes를 로드하여 export만 수행
#
# 여기서는 옵션 A (실행 + export) 를 기본으로 합니다.
# 이미 결과가 있으면 DO_SOLVE=False로 설정하세요.

DO_SOLVE = True
PHASE_ADVANCE = 43.33
RMS_CURRENT = 460  # Ref model

# FEA export 설정
FIRST_STEP = 1
FINAL_STEP = mcad.get_variable("TorquePointsPerCycle")  # Motor-CAD 기본 TorquePointsPerCycle 정도
EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

out_root = Path(mcad_default_export_dir(mcad))
export_dir = out_root / "ACLossCalcExport"
export_dir.mkdir(parents=True, exist_ok=True)


# [3] 90-Point FEA Sweep & Directory Backup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [3] 180-Point FEA Sweep (Hybrid & FullFEA) & Directory Backup
# ─────────────────────────────────────────────────────────────────────────────
import os
import shutil
import json
from pathlib import Path
from datetime import datetime
from scipy.io import savemat

# 90-Point Sweep Definition (Total 180 points for both ProximityLossModels)
CURRENT_LIST = np.linspace(0.1, 460.0, 5)   # 5 currents
PHASE_LIST = np.linspace(0.0, 90.0, 6)      # 6 phase angles
PROXIMITY_MODELS = [1, 3]                  # 1: Hybrid, 3: FullFEA (TS)

# FEA export settings
FIRST_STEP = 1
EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

out_root = Path(mcad_default_export_dir(mcad))
backup_root = out_root / "ACLossCalcExport_Map"
backup_root.mkdir(parents=True, exist_ok=True)

sweep_results = []
total_points = len(SPEED_LIST) * len(CURRENT_LIST) * len(PHASE_LIST) * len(PROXIMITY_MODELS)
point_idx = 0

print(f"Starting 180-point sweep (speeds: {SPEED_LIST}, currents: {list(np.round(CURRENT_LIST, 1))}, phases: {list(np.round(PHASE_LIST, 1))})...")
print(f"Backup root: {backup_root}\n")

def calc_dc_loss_kw(resistance_ohm: float, rms_current_a: float) -> float:
    """3상 DC 손실 [kW] = 3 * R * I²."""
    return 3.0 * resistance_ohm * (rms_current_a ** 2) / 1000.0

# Get resistances once to calculate DC loss for FullFEA
try:
    R_total = float(mcad.get_variable("Resistance_MotorLAB")) * 4.0
    R_end = float(mcad.get_variable("EndWindingResistance_Lab")) * 4.0
    R_active = R_total - R_end
except Exception as e:
    R_total, R_end, R_active = 0.0, 0.0, 0.0
    print(f"  [WARN] Failed to read winding resistances: {e}")

for prox_model in PROXIMITY_MODELS:
    mcad.set_variable("ProximityLossModel", prox_model)
    mode_label = "Hybrid" if prox_model == 1 else "FullFEA"
    
    for speed in SPEED_LIST:
        mcad.set_variable("ShaftSpeed", speed)
        for current in CURRENT_LIST:
            mcad.set_variable("RMSCurrent", current)
            for phase in PHASE_LIST:
                mcad.set_variable("PhaseAdvance", phase)
                
                point_idx += 1
                print(f"[{point_idx}/{total_points}] [{mode_label}] Speed: {speed} RPM, Current: {current:.1f} A, Phase: {phase:.1f} deg")
                
                # 1. Run calculation
                print(f"  → Solving {mode_label} FEA...")
                mcad.do_magnetic_calculation()
                
                # Get TorquePointsPerCycle for TS or just use default
                torque_points = int(mcad.get_variable("TorquePointsPerCycle"))
                
                # 2. Get latest solved results directory
                try:
                    latest_mes = find_latest_mes(mcad)
                    active_results_dir = latest_mes.parent
                except Exception as e:
                    print(f"  [ERROR] Failed to locate latest .mes file: {e}")
                    continue
                
                # 3. Create destination folder
                point_folder_name = f"{mode_label}_Speed_{speed}RPM_{current:.1f}A_{phase:.1f}deg"
                dest_point_dir = backup_root / point_folder_name
                dest_results_dir = dest_point_dir / "FEResultsData"
                
                # 4. Copy active results folder
                print(f"  → Backing up results folder to: {point_folder_name}/FEResultsData")
                if dest_results_dir.exists():
                    shutil.rmtree(dest_results_dir)
                shutil.copytree(active_results_dir, dest_results_dir)
                
                # 5. Export B-field TXT file to the destination directory
                txt_path = dest_point_dir / "FEA_data.txt"
                print(f"  → Exporting B-field TXT to: {point_folder_name}/FEA_data.txt")
                mcad.save_fea_data(str(txt_path), FIRST_STEP, torque_points, EXPORT_COLUMNS, "", ",")
                
                # 6. Read losses and prepare point summary
                point_data = {
                    "proximity_model": prox_model,
                    "mode": mode_label,
                    "speed": speed,
                    "current": current,
                    "phase": phase,
                    "backup_dir": str(dest_point_dir)
                }
                
                if prox_model == 1:
                    # Hybrid scalar losses
                    try:
                        total_w = float(mcad.get_variable("ACLoss_Hybrid_Total"))
                        prox_w = float(mcad.get_variable("ACLoss_Hybrid_Prox_Total"))
                        skin_w = float(mcad.get_variable("ACLoss_Hybrid_SkinEffect_Total"))
                    except Exception as e:
                        total_w, prox_w, skin_w = 0.0, 0.0, 0.0
                        print(f"  [WARN] Failed to read hybrid losses: {e}")
                    point_data.update({
                        "hybrid_total_W": total_w,
                        "hybrid_prox_W": prox_w,
                        "hybrid_skin_W": skin_w,
                        "hybrid_total_kW": total_w / 1000.0,
                        "hybrid_prox_kW": prox_w / 1000.0,
                        "hybrid_skin_kW": skin_w / 1000.0,
                    })
                    print(f"  → Hybrid Loss: Total={total_w:.1f} W, Prox={prox_w:.1f} W, Skin={skin_w:.1f} W\n")
                else:
                    # FullFEA / TS scalar losses
                    try:
                        per_turn_str = mcad.get_variable("ACLoss_FEA_OnLoad_PerTurn")
                        if isinstance(per_turn_str, str):
                            per_turn_w = [float(x) for x in per_turn_str.split(":")]
                        else:
                            per_turn_w = list(per_turn_str)
                        per_turn_sum_kw = sum(per_turn_w) / 1000.0
                        total_kw = float(mcad.get_variable("ACLoss_FEA_OnLoad_Total")) / 1000.0
                    except Exception as e:
                        per_turn_w = []
                        per_turn_sum_kw, total_kw = 0.0, 0.0
                        print(f"  [WARN] Failed to read TS losses: {e}")
                    
                    dc_active_kw = calc_dc_loss_kw(R_active, current)
                    dc_end_kw = calc_dc_loss_kw(R_end, current)
                    ac_active_only_kw = per_turn_sum_kw - dc_active_kw
                    
                    point_data.update({
                        "ts_per_turn_W": per_turn_w,
                        "ts_per_turn_sum_kW": per_turn_sum_kw,
                        "ts_total_kW": total_kw,
                        "ts_dc_active_kW": dc_active_kw,
                        "ts_dc_end_kW": dc_end_kw,
                        "ts_ac_active_only_kW": ac_active_only_kw,
                    })
                    print(f"  → TS Loss: PerTurnSum={per_turn_sum_kw:.3f} kW, AC Active Only={ac_active_only_kw:.3f} kW, Total={total_kw:.3f} kW\n")
                
                sweep_results.append(point_data)

print(f"\n✓ Sweep complete! {point_idx} points processed and backed up under {backup_root}")

# ─────────────────────────────────────────────────────────────────────────────
# Save extracted scalar data to MAT and JSON
# ─────────────────────────────────────────────────────────────────────────────
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path("map_exports")
out_dir.mkdir(parents=True, exist_ok=True)
json_path = out_dir / f"JEET_ACLoss_180Map_Summary_{ts}.json"
mat_path = out_dir / f"JEET_ACLoss_180Map_Summary_{ts}.mat"

# Save JSON
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(sweep_results, f, ensure_ascii=False, indent=2)
print(f"Saved JSON summary: {json_path}")

# Save MAT (matlab compatible format)
hybrid_pts = [p for p in sweep_results if p["proximity_model"] == 1]
ts_pts = [p for p in sweep_results if p["proximity_model"] == 3]

def _arr(v):
    a = np.array(v, dtype=np.float64)
    return a.reshape(-1, 1)

mat_data = {
    "speeds_RPM": _arr([p["speed"] for p in hybrid_pts]),
    "currents_A": _arr([p["current"] for p in hybrid_pts]),
    "phases_deg": _arr([p["phase"] for p in hybrid_pts]),
    
    "hybrid_Total_kW": _arr([p["hybrid_total_kW"] for p in hybrid_pts]),
    "hybrid_Prox_kW": _arr([p["hybrid_prox_kW"] for p in hybrid_pts]),
    "hybrid_Skin_kW": _arr([p["hybrid_skin_kW"] for p in hybrid_pts]),
    
    "ts_OnLoad_PerTurnSum_kW": _arr([p["ts_per_turn_sum_kW"] for p in ts_pts]),
    "ts_Total_kW": _arr([p["ts_total_kW"] for p in ts_pts]),
    "ts_DC_Active_kW": _arr([p["ts_dc_active_kW"] for p in ts_pts]),
    "ts_DC_End_kW": _arr([p["ts_dc_end_kW"] for p in ts_pts]),
    "ts_ActiveOnly_kW": _arr([p["ts_ac_active_only_kW"] for p in ts_pts]),
}

if ts_pts and ts_pts[0].get("ts_per_turn_W"):
    mat_data["ts_OnLoad_PerTurn_kW"] = np.array([p["ts_per_turn_W"] for p in ts_pts], dtype=np.float64) / 1000.0

savemat(str(mat_path), mat_data, do_compression=True)
print(f"Saved MATLAB MAT: {mat_path}")


# [3c] 보완 스윕: 8000 RPM 추가 (4×4×4 맵 완성)

기존 [2000, 4000, 16000] RPM 데이터에 **8000 RPM**만 추가합니다.

| 항목 | 기존 | 추가 (8000 RPM) |
|---|---|---|
| 전류 | 5개 (0.1 ~ 460 A) | **4개** (near-zero 제외: 115, 230, 345, 460 A) |
| 위상 | 6개 (0 ~ 90°) | **4개** (0, 18, 54, 90°) |
| FEA 실행 | 180회 완료 | **+32회** (4×4×2모델) |

완료 후 JSON을 병합 저장하여 이후 셀에서 4속도 데이터로 사용합니다.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# [3c] 보완 스윕: 8000 RPM 추가
# 기존 JSON 로드 → 8000 RPM 실행 → 병합 저장
# ─────────────────────────────────────────────────────────────────────────────
import json, shutil, glob
import numpy as np
from pathlib import Path
from datetime import datetime
from tools.motorCAD.pyMCAD import calc_dc_loss_kw

ADDON_SPEED = 8000  # 추가할 속도 [RPM]

# 셀 [3]이 실행되지 않은 경우 기본값 정의
if 'CURRENT_LIST' not in globals():
    CURRENT_LIST = np.linspace(0.1, 460.0, 5)
if 'PHASE_LIST' not in globals():
    PHASE_LIST = np.linspace(0.0, 90.0, 6)
if 'PROXIMITY_MODELS' not in globals():
    PROXIMITY_MODELS = [1, 3]
if 'FIRST_STEP' not in globals():
    FIRST_STEP = 1
if 'EXPORT_COLUMNS' not in globals():
    EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

# 기존 5전류 × 6위상 그리드에서 4×4 서브셋 선택
#   전류: near-zero(0.1 A) 제외 → 인덱스 [1,2,3,4]
#   위상: 균등 4개 → 인덱스 [0,1,3,5] = [0, 18, 54, 90] deg
ADDON_CURRENT_LIST = CURRENT_LIST[1:]          # [115.1, 230.1, 345.1, 460.0] A
ADDON_PHASE_LIST   = PHASE_LIST[[0, 1, 3, 5]]  # [0.0, 18.0, 54.0, 90.0] deg

print("=== 8000 RPM 보완 스윕 ===")
print(f"  전류: {np.round(ADDON_CURRENT_LIST, 1).tolist()} A  ({len(ADDON_CURRENT_LIST)}개)")
print(f"  위상: {ADDON_PHASE_LIST.tolist()} deg  ({len(ADDON_PHASE_LIST)}개)")
print(f"  모델: Hybrid(1) + FullFEA(3)")
total_addon = len(ADDON_CURRENT_LIST) * len(ADDON_PHASE_LIST) * len(PROXIMITY_MODELS)
print(f"  FEA 실행 예정: {total_addon}회\n")

# ── 기존 JSON 로드 ────────────────────────────────────────────────────────────
json_files = sorted(glob.glob("map_exports/JEET_ACLoss_180Map_Summary_*.json"))
if json_files:
    with open(json_files[-1], "r", encoding="utf-8") as f:
        sweep_results = json.load(f)
    print(f"기존 데이터 로드: {json_files[-1]}")
    print(f"  기존 포인트: {len(sweep_results)}개, "
          f"속도: {sorted(set(p['speed'] for p in sweep_results))} RPM\n")
elif 'sweep_results' not in globals():
    raise RuntimeError("기존 sweep 데이터 없음. 먼저 셀 [3b]를 실행하세요.")

# ── 저항값 읽기 (셀 [3] 미실행 시 여기서 직접 읽음) ─────────────────────────
if 'R_active' not in globals() or 'R_end' not in globals():
    try:
        _R_total = float(mcad.get_variable("Resistance_MotorLAB")) * 4.0
        R_end    = float(mcad.get_variable("EndWindingResistance_Lab")) * 4.0
        R_active = _R_total - R_end
        print(f"저항값 읽기 완료: R_active={R_active:.6f} Ω, R_end={R_end:.6f} Ω\n")
    except Exception as e:
        R_active = R_end = 0.0
        print(f"  [WARN] 저항값 읽기 실패: {e} → DC 손실 = 0으로 처리\n")

# ── 중복 방지: 이미 8000 RPM 있으면 스킵 ─────────────────────────────────────
existing_8k = [p for p in sweep_results if p["speed"] == ADDON_SPEED]
if existing_8k:
    print(f"이미 {ADDON_SPEED} RPM 데이터 {len(existing_8k)}개 존재 → 스킵")
else:
    out_root    = Path(mcad_default_export_dir(mcad))
    backup_root = out_root / "ACLossCalcExport_Map"
    backup_root.mkdir(parents=True, exist_ok=True)

    pt_idx = 0
    mcad.set_variable("ShaftSpeed", ADDON_SPEED)

    for prox_model in PROXIMITY_MODELS:
        mcad.set_variable("ProximityLossModel", prox_model)
        mode_label = "Hybrid" if prox_model == 1 else "FullFEA"

        for current in ADDON_CURRENT_LIST:
            mcad.set_variable("RMSCurrent", current)
            for phase in ADDON_PHASE_LIST:
                mcad.set_variable("PhaseAdvance", phase)
                pt_idx += 1
                print(f"[{pt_idx}/{total_addon}] [{mode_label}] "
                      f"{ADDON_SPEED} RPM, {current:.1f} A, {phase:.1f}°")

                mcad.do_magnetic_calculation()
                torque_points = int(mcad.get_variable("TorquePointsPerCycle"))

                try:
                    latest_mes      = find_latest_mes(mcad) # 최근 .mes 파일 경로
                    active_res_dir  = latest_mes.parent # FEA 결과 폴더 경로
                except Exception as e:
                    print(f"  [ERROR] {e}")
                    continue

                folder  = f"{mode_label}_Speed_{ADDON_SPEED}RPM_{current:.1f}A_{phase:.1f}deg" # FEA 결과 폴더 이름
                dst_fe  = backup_root / folder / "FEResultsData" # FEA 결과 폴더 경로
                if dst_fe.exists(): shutil.rmtree(dst_fe) # 기존 FEA 결과 삭제
                shutil.copytree(active_res_dir, dst_fe) # FEA 결과 폴더 복사

                txt_path = backup_root / folder / "FEA_data.txt" # FEA 데이터 파일 경로
                mcad.save_fea_data(str(txt_path), FIRST_STEP, torque_points, EXPORT_COLUMNS, "", ",")
                
                # FEA 데이터 포인트 생성
                pt = {"proximity_model": prox_model, "mode": mode_label,
                      "speed": ADDON_SPEED, "current": current, "phase": phase,
                      "backup_dir": str(backup_root / folder)}

                if prox_model == 1:
                    try:
                        tw = float(mcad.get_variable("ACLoss_Hybrid_Total"))
                        pw = float(mcad.get_variable("ACLoss_Hybrid_Prox_Total"))
                        sw = float(mcad.get_variable("ACLoss_Hybrid_SkinEffect_Total"))
                    except:
                        tw = pw = sw = 0.0
                    pt.update({
                        "hybrid_total_W": tw, "hybrid_prox_W": pw, "hybrid_skin_W": sw,
                        "hybrid_total_kW": tw/1e3, "hybrid_prox_kW": pw/1e3,
                        "hybrid_skin_kW": sw/1e3,
                    })
                    print(f"  → Hybrid Total: {tw:.1f} W\n")
                else:
                    try:
                        pt_str = mcad.get_variable("ACLoss_FEA_OnLoad_PerTurn")
                        ptw    = [float(x) for x in pt_str.split(":")] \
                                 if isinstance(pt_str, str) else list(pt_str)
                        pts_kw = sum(ptw) / 1e3
                        tot_kw = float(mcad.get_variable("ACLoss_FEA_OnLoad_Total")) / 1e3
                    except:
                        ptw, pts_kw, tot_kw = [], 0., 0.
                    dc_act = calc_dc_loss_kw(R_active, current)
                    dc_end = calc_dc_loss_kw(R_end, current)
                    ac_act = pts_kw - dc_act
                    pt.update({
                        "ts_per_turn_W": ptw, "ts_per_turn_sum_kW": pts_kw,
                        "ts_total_kW": tot_kw, "ts_dc_active_kW": dc_act,
                        "ts_dc_end_kW": dc_end, "ts_ac_active_only_kW": ac_act,
                    })
                    print(f"  → FullFEA AC Active Only: {ac_act:.3f} kW\n")

                sweep_results.append(pt)

    # ── 병합 저장 ─────────────────────────────────────────────────────────────
    ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = Path("map_exports")
    save_path = out_dir / f"JEET_ACLoss_4Speed_Map_Summary_{ts}.json"
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(sweep_results, f, ensure_ascii=False, indent=2)

    spds = sorted(set(p["speed"] for p in sweep_results))
    print(f"\n✓ 병합 완료: 총 {len(sweep_results)}포인트 | 속도: {spds} RPM")
    print(f"✓ 저장: {save_path}")

# ── 스윕 구성 확인 ────────────────────────────────────────────────────────────
print("\n=== 최종 데이터 구성 ===")
for spd in sorted(set(p["speed"] for p in sweep_results)):
    h_pts = [p for p in sweep_results if p["speed"]==spd and p["proximity_model"]==1]
    f_pts = [p for p in sweep_results if p["speed"]==spd and p["proximity_model"]==3]
    print(f"  {spd:5d} RPM → Hybrid: {len(h_pts):2d}pt, FullFEA: {len(f_pts):2d}pt")

=== 8000 RPM 보완 스윕 ===
  전류: [115.1, 230.0, 345.0, 460.0] A  (4개)
  위상: [0.0, 18.0, 54.0, 90.0] deg  (4개)
  모델: Hybrid(1) + FullFEA(3)
  FEA 실행 예정: 32회

기존 데이터 로드: map_exports\JEET_ACLoss_180Map_Summary_20260620_055628.json
  기존 포인트: 180개, 속도: [2000, 4000, 16000] RPM

저항값 읽기 완료: R_active=0.087978 Ω, R_end=0.071417 Ω

[1/32] [Hybrid] 8000 RPM, 115.1 A, 0.0°
  → Hybrid Total: 930.9 W

[2/32] [Hybrid] 8000 RPM, 115.1 A, 18.0°
  → Hybrid Total: 1021.8 W

[3/32] [Hybrid] 8000 RPM, 115.1 A, 54.0°
  → Hybrid Total: 1080.3 W

[4/32] [Hybrid] 8000 RPM, 115.1 A, 90.0°
  → Hybrid Total: 1158.6 W

[5/32] [Hybrid] 8000 RPM, 230.0 A, 0.0°
  → Hybrid Total: 3055.7 W

[6/32] [Hybrid] 8000 RPM, 230.0 A, 18.0°
  → Hybrid Total: 3530.8 W

[7/32] [Hybrid] 8000 RPM, 230.0 A, 54.0°
  → Hybrid Total: 4282.6 W

[8/32] [Hybrid] 8000 RPM, 230.0 A, 90.0°
  → Hybrid Total: 4421.7 W

[9/32] [Hybrid] 8000 RPM, 345.0 A, 0.0°
  → Hybrid Total: 5765.6 W

[10/32] [Hybrid] 8000 RPM, 345.0 A, 18.0°
  → Hybrid Total: 6757

# [4] id-iq 평면 AC Active Only 손실 Surface 플롯 (속도별)

ProximityLossModel = 1(Hybrid) 및 3(FullFEA/TS) 각각에 대해 속도별로 $I_d, I_q$ 평면에서의 AC Active Only 손실 Surface 플롯을 시각화합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [4] id-iq 평면 AC Active Only 손실 Surface 플롯 (Hybrid vs FullFEA 대화형 비교)
# ─────────────────────────────────────────────────────────────────────────────
# [Matplotlib 백엔드 설정]
# - 'auto'     : 환경 자동 감지 (VS Code -> widget, 브라우저 -> inline)
# - 'inline'   : 정적 이미지 출력 (웹 브라우저 JupyterLab에서 렌더링 에러 발생 시 이 값으로 설정하세요!)
# - 'widget'   : VS Code 및 JupyterLab용 대화형 플롯 (ipympl 필요)
# - 'notebook' : 클래식 Jupyter Notebook용 대화형 플롯 (nbagg)
PLOT_BACKEND = 'auto'  # <-- 브라우저에서 플롯이 안 뜨거나 렌더링 에러가 나면 'inline'으로 변경하고 다시 실행하세요!

import os
import sys
import glob
import json
from pathlib import Path

try:
    import IPython
    shell = IPython.get_ipython()
    if shell is not None:
        has_vscode_env = any(k.startswith('VSCODE_') for k in os.environ.keys())
        has_vscode_modules = any('vscode' in m.lower() for m in sys.modules.keys())
        
        selected_backend = PLOT_BACKEND
        if selected_backend == 'auto':
            if has_vscode_env and has_vscode_modules:
                selected_backend = 'widget'
            else:
                selected_backend = 'inline'
        
        print("--- Matplotlib Backend Config ---")
        print(f"  [Selection] '{PLOT_BACKEND}' -> '{selected_backend}'")
        
        if selected_backend == 'widget':
            shell.run_line_magic('matplotlib', 'widget')
        elif selected_backend == 'notebook':
            shell.run_line_magic('matplotlib', 'notebook')
        else:
            shell.run_line_magic('matplotlib', 'inline')
except Exception as e:
    print(f"Failed to set matplotlib backend: {e}")

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as mpatches

# ── 항상 JSON에서 최신 데이터 로드 (4Speed 우선, 180Map 폴백) ────────────────
_json_candidates = (
    sorted(glob.glob("map_exports/JEET_ACLoss_4Speed_Map_Summary_*.json"))
    or sorted(glob.glob("map_exports/JEET_ACLoss_180Map_Summary_*.json"))
)
if _json_candidates:
    _load_path = _json_candidates[-1]
    print(f"[데이터 로드] {_load_path}")
    with open(_load_path, "r", encoding="utf-8") as f:
        sweep_results = json.load(f)
    # 모델 검증: 모든 backup_dir이 SLFEA_Half 경로인지 확인
    _non_half = [p for p in sweep_results if "backup_dir" in p and "SLFEA_Half" not in p["backup_dir"]]
    if _non_half:
        print(f"  [WARNING] {len(_non_half)}개 포인트가 SLFEA_Half 모델이 아닙니다!")
        print(f"  예: {_non_half[0]['backup_dir']}")
    else:
        print(f"  [OK] 전체 {len(sweep_results)}포인트 SLFEA_Half 모델 확인")
    _speeds = sorted(set(p["speed"] for p in sweep_results))
    print(f"  속도: {_speeds} RPM, 총 {len(sweep_results)}포인트")
else:
    print("[ERROR] map_exports/ 에서 JSON 파일을 찾을 수 없습니다!")
    sweep_results = []

if sweep_results:
    hybrid_data = [p for p in sweep_results if p["proximity_model"] == 1]
    ts_data = [p for p in sweep_results if p["proximity_model"] == 3]
    
    def process_pts(pts, is_hybrid):
        speeds = np.array([p["speed"] for p in pts])
        currents = np.array([p["current"] for p in pts])
        phases = np.array([p["phase"] for p in pts])
        
        amplitude = currents * np.sqrt(2)
        phase_rad = (phases + 90) * np.pi / 180.0
        id_vals = amplitude * np.cos(phase_rad)
        iq_vals = amplitude * np.sin(phase_rad)
        
        if is_hybrid:
            losses = np.array([p["hybrid_total_kW"] for p in pts])
        else:
            losses = np.array([p["ts_ac_active_only_kW"] for p in pts])
            
        return speeds, id_vals, iq_vals, losses, pts

    speed_colors = {2000: 'cyan', 4000: 'limegreen', 8000: 'orange', 16000: 'tomato'}
    default_colors = ['cyan', 'limegreen', 'orange', 'tomato']
    
    def create_interactive_comparison_plot(pts_hybrid, pts_ts):
        speeds_h, id_h, iq_h, losses_h, raw_h = process_pts(pts_hybrid, is_hybrid=True)
        speeds_f, id_f, iq_f, losses_f, raw_f = process_pts(pts_ts, is_hybrid=False)
        
        currents_h = np.array([p["current"] for p in raw_h])
        phases_h = np.array([p["phase"] for p in raw_h])
        currents_f = np.array([p["current"] for p in raw_f])
        phases_f = np.array([p['phase'] for p in raw_f])
        
        unique_speeds = sorted(list(set(speeds_h)))
        
        fig = plt.figure(figsize=(18, 5.5))
        fig.suptitle("AC Loss Comparison Map: Hybrid vs FullFEA (HalfSC Model)", fontsize=13, fontweight='bold')
        
        ax_left = fig.add_subplot(131, projection='3d')
        ax_left.set_title("Hybrid (ProximityLossModel = 1)", fontsize=11, fontweight='bold')
        ax_mid = fig.add_subplot(132, projection='3d')
        ax_mid.set_title("FullFEA (ProximityLossModel = 3)", fontsize=11, fontweight='bold')
        
        legend_patches_h = []
        legend_patches_f = []
        
        for i, spd in enumerate(unique_speeds):
            color = speed_colors.get(spd, default_colors[i % len(default_colors)])
            idx_h = (speeds_h == spd)
            if np.any(idx_h):
                ax_left.plot_trisurf(id_h[idx_h], iq_h[idx_h], losses_h[idx_h], color=color, edgecolor='none', alpha=0.35)
                legend_patches_h.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
            idx_f = (speeds_f == spd)
            if np.any(idx_f):
                ax_mid.plot_trisurf(id_f[idx_f], iq_f[idx_f], losses_f[idx_f], color=color, edgecolor='none', alpha=0.35)
                legend_patches_f.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
                
        sc_h = ax_left.scatter(id_h, iq_h, losses_h, c='grey', s=25, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
        sc_f = ax_mid.scatter(id_f, iq_f, losses_f, c='grey', s=25, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
        
        for ax, lp in [(ax_left, legend_patches_h), (ax_mid, legend_patches_f)]:
            ax.set_xlabel("I_d [A]", fontsize=8, labelpad=7)
            ax.set_ylabel("I_q [A]", fontsize=8, labelpad=7)
            ax.set_zlabel("AC Loss [kW]", fontsize=8, labelpad=7)
            ax.legend(handles=lp, fontsize=8, loc="upper right")
            
        ax_right = fig.add_subplot(133)
        ax_right.text(0.5, 0.5, "Click any point in left/middle 3D plots\nand press 'Space' to draw comparison curves", 
                     ha="center", va="center", fontsize=10, color="gray")
        ax_right.set_xlabel("Speed [RPM]", fontsize=9)
        ax_right.set_ylabel("AC Loss [kW]", fontsize=9)
        ax_right.grid(True, linestyle="--", alpha=0.5)
        
        selected_pt = {"current": None, "phase": None, "id": None, "iq": None}
        highlights_h = []
        highlights_f = []
        
        annotation_h = ax_left.text2D(0.02, 0.95, "", transform=ax_left.transAxes, 
                                      bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
        annotation_f = ax_mid.text2D(0.02, 0.95, "", transform=ax_mid.transAxes, 
                                     bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
        annotation_h.set_visible(False)
        annotation_f.set_visible(False)
        
        def on_pick(event):
            if event.artist not in [sc_h, sc_f]:
                return
            idx = event.ind[0]
            if event.artist == sc_h:
                curr, ph = raw_h[idx]["current"], raw_h[idx]["phase"]
            else:
                curr, ph = raw_f[idx]["current"], raw_f[idx]["phase"]
            selected_pt["current"] = curr
            selected_pt["phase"] = ph
            amp = curr * np.sqrt(2)
            phase_rad = (ph + 90) * np.pi / 180.0
            selected_pt["id"] = amp * np.cos(phase_rad)
            selected_pt["iq"] = amp * np.sin(phase_rad)
            for h in highlights_h + highlights_f:
                h.remove()
            highlights_h.clear()
            highlights_f.clear()
            same_h_idx = (currents_h == curr) & (phases_h == ph)
            hh = ax_left.scatter(id_h[same_h_idx], iq_h[same_h_idx], losses_h[same_h_idx], 
                                 color='red', s=70, edgecolors='black', linewidths=1.8, zorder=10)
            highlights_h.append(hh)
            same_f_idx = (currents_f == curr) & (phases_f == ph)
            hf = ax_mid.scatter(id_f[same_f_idx], iq_f[same_f_idx], losses_f[same_f_idx], 
                                color='red', s=70, edgecolors='black', linewidths=1.8, zorder=10)
            highlights_f.append(hf)
            msg = (f"Selected: I_rms={curr:.1f}A, Phase={ph:.1f}°\n"
                   f"Id={selected_pt['id']:.1f}A, Iq={selected_pt['iq']:.1f}A\n→ Press 'Space'")
            for annot in [annotation_h, annotation_f]:
                annot.set_text(msg)
                annot.set_visible(True)
            fig.canvas.draw_idle()
            
        def on_key(event):
            if event.key != ' ' or selected_pt["current"] is None:
                return
            ax_right.clear()
            curr, ph = selected_pt["current"], selected_pt["phase"]
            curve_speeds, curve_losses_h, curve_losses_f = [], [], []
            for spd in unique_speeds:
                match_h = [p for p in raw_h if p["speed"] == spd and np.isclose(p["current"], curr) and np.isclose(p["phase"], ph)]
                match_f = [p for p in raw_f if p["speed"] == spd and np.isclose(p["current"], curr) and np.isclose(p["phase"], ph)]
                if match_h and match_f:
                    curve_speeds.append(spd)
                    curve_losses_h.append(match_h[0]["hybrid_total_kW"])
                    curve_losses_f.append(match_f[0]["ts_ac_active_only_kW"])
            ax_right.plot(curve_speeds, curve_losses_h, marker='o', linestyle='-', color='dodgerblue', linewidth=2, label="Hybrid AC Total")
            ax_right.plot(curve_speeds, curve_losses_f, marker='*', linestyle='--', color='crimson', linewidth=2, label="FullFEA AC Active Only")
            for xs, yh, yf in zip(curve_speeds, curve_losses_h, curve_losses_f):
                ax_right.annotate(f"{yh:.2f}", xy=(xs, yh), xytext=(4, 4), textcoords="offset points", fontsize=8, color="dodgerblue")
                ax_right.annotate(f"{yf:.2f}", xy=(xs, yf), xytext=(4, -12), textcoords="offset points", fontsize=8, color="crimson")
            ax_right.set_title(f"AC Loss vs Speed\n(I_rms={curr:.1f}A, Phase={ph:.1f}°)", fontsize=11, fontweight='bold')
            ax_right.set_xlabel("Speed [RPM]", fontsize=9)
            ax_right.set_ylabel("AC Loss [kW]", fontsize=9)
            ax_right.grid(True, linestyle="--", alpha=0.5)
            ax_right.legend(fontsize=9, loc="upper left")
            fig.canvas.draw_idle()
            
        fig.canvas.mpl_connect('pick_event', on_pick)
        fig.canvas.mpl_connect('key_press_event', on_key)
        plt.tight_layout()
        plt.show()
        
    if len(hybrid_data) > 0 and len(ts_data) > 0:
        create_interactive_comparison_plot(hybrid_data, ts_data)
    else:
        print("Error: Need both Hybrid and FullFEA data.")
else:
    print("No sweep results found to plot.")


# [5] Adjustment Factor (AF) 모델링

`AF = FullFEA_AC_active_only / Hybrid_AC_total`을 **(speed, Irms, phase)** 기반으로 모델링합니다.

**AF에 영향을 주는 물리적 인자:**
- **Skin effect**: 고속(고주파수) 영역에서 주파수 자승 비선형성이 꺾이는 와전류 차폐(Back-reaction) 경향을 보정합니다.
- **Proximity effect**: 고정자 전류 크기($I_{rms}$) 및 고정자 전류와 회전자 극 간의 상대 위상각($\theta$)에 의한 AC 손실 비선형 곡면을 피팅합니다.
- **Id-Iq 결합**:MTPA 운전 궤적 및 field-weakening 영역에서의 AC 손실 거동을 전류 크기와 위상각을 통해 정합시킵니다.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [5] Adjustment Factor (AF) Map: FullFEA / Hybrid 비율 계산
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import json
import glob
from pathlib import Path

# ── 항상 JSON에서 최신 데이터 로드 (4Speed 우선, 180Map 폴백) ────────────────
_json_candidates = (
    sorted(glob.glob("map_exports/JEET_ACLoss_4Speed_Map_Summary_*.json"))
    or sorted(glob.glob("map_exports/JEET_ACLoss_180Map_Summary_*.json"))
)
if _json_candidates:
    _load_path = _json_candidates[-1]
    print(f"[데이터 로드] {_load_path}")
    with open(_load_path, "r", encoding="utf-8") as f:
        sweep_results = json.load(f)
    # 모델 검증: 모든 backup_dir이 SLFEA_Half 경로인지 확인
    _non_half = [p for p in sweep_results if "backup_dir" in p and "SLFEA_Half" not in p["backup_dir"]]
    if _non_half:
        print(f"  [WARNING] {len(_non_half)}개 포인트가 SLFEA_Half 모델이 아닙니다!")
        print(f"  예: {_non_half[0]['backup_dir']}")
    else:
        print(f"  [OK] 전체 {len(sweep_results)}포인트 SLFEA_Half 모델 확인")
    _speeds = sorted(set(p["speed"] for p in sweep_results))
    print(f"  속도: {_speeds} RPM, 총 {len(sweep_results)}포인트")
else:
    raise RuntimeError("map_exports/ 에서 JSON 파일을 찾을 수 없습니다! 셀 [3] 또는 [3c]를 먼저 실행하세요.")

hybrid_data = [p for p in sweep_results if p["proximity_model"] == 1]
ts_data     = [p for p in sweep_results if p["proximity_model"] == 3]

# (speed, current, phase) 기준 매칭 → AF 계산
af_points = []
for ts_pt in ts_data:
    spd  = ts_pt["speed"]
    curr = ts_pt["current"]
    ph   = ts_pt["phase"]

    matches = [p for p in hybrid_data
               if p["speed"] == spd
               and np.isclose(p["current"], curr, atol=1e-3)
               and np.isclose(p["phase"],   ph,   atol=1e-3)]
    if not matches:
        continue
    h_pt = matches[0]

    h_ac = h_pt["hybrid_total_kW"]
    f_ac = ts_pt["ts_ac_active_only_kW"]

    if h_ac < 1e-4:          # 전류 거의 0 → skip (분모 불안정)
        continue

    af = f_ac / h_ac

    # dq 변환 (peak)
    amp    = curr * np.sqrt(2)
    ph_rad = (ph + 90.0) * np.pi / 180.0
    id_a   = amp * np.cos(ph_rad)
    iq_a   = amp * np.sin(ph_rad)

    af_points.append({
        "speed_rpm":   spd,
        "speed_kRPM":  spd / 1000.0,
        "current_rms": curr,
        "phase_deg":   ph,
        "id_A":        id_a,
        "iq_A":        iq_a,
        "hybrid_ac_kW": h_ac,
        "fea_ac_kW":    f_ac,
        "AF":           af,
    })

print(f"\nAF 계산 완료: {len(af_points)}개 운전점\n")
print(f"{'Speed[kRPM]':>12} {'Id[A]':>9} {'Iq[A]':>9} {'H_AC[kW]':>11} {'F_AC[kW]':>11} {'AF[-]':>7}")
print("─" * 65)
for p in af_points:
    print(f"{p['speed_kRPM']:>12.1f} {p['id_A']:>9.1f} {p['iq_A']:>9.1f} "
          f"{p['hybrid_ac_kW']:>11.3f} {p['fea_ac_kW']:>11.3f} {p['AF']:>7.3f}")

af_arr = np.array([p["AF"] for p in af_points])
print(f"\nAF 통계: min={af_arr.min():.3f}, max={af_arr.max():.3f}, "
      f"mean={af_arr.mean():.3f}, std={af_arr.std():.3f}")

# [5.5] 방법 B: RBF 모델 비교 (3D TPS RBF vs. 1D x 2D 차원 분리형 RBF)

이 단계에서는 두 가지 유형의 글로벌 Thin-Plate Spline (TPS) RBF 대리 모델을 동시 수립합니다:

1. **3D TPS RBF 모델 (Full Interpolation)**:
   - 속도, 전류, 위상각 3차원 입력에 대해 106개 데이터 포인트를 모두 RBF 센터로 삼아 완벽히 매칭하는 모델입니다. 
   - 훈련 오차는 0%에 수렴하지만, Motor-CAD Lab 식의 길이가 다소 길어집니다 (~28k 캐릭터).

2. **1D x 2D 차원 분리형 스케일링 모델 (Separable Scaling Model)**:
   - 단일 속도(2.0 kRPM)의 30개 점으로 2D TPS RBF인 $g(I, \theta)$ 형상을 먼저 피팅하고, 속도 증가에 따른 스케일링 배율 $f(speed)$을 다른 속도 영역의 12개 대표 점으로 평균/2차 다항식 피팅하는 모델입니다.
   - 수식이 30개 항으로 압축되어 매우 슬림하고 (~5.5k 캐릭터), 데이터 공백에서의 과적합 없이 안정적으로 동작합니다.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [5.5] 방법 B: 두 가지 RBF 모델 동시 수립 (3D TPS RBF 및 1D x 2D Separable RBF)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

# ── 데이터 준비 ──────────────────────────────────────────────────────────────
speeds_k  = np.array([p["speed_kRPM"]  for p in af_points])
irms_arr  = np.array([p["current_rms"] for p in af_points])   # Irms [A]
phase_arr = np.array([p["phase_deg"]   for p in af_points])   # phase advance [deg]
af_arr    = np.array([p["AF"]          for p in af_points])
id_arr    = np.array([p["id_A"]        for p in af_points])
iq_arr    = np.array([p["iq_A"]        for p in af_points])
curr_arr  = irms_arr.copy()
X_data    = np.column_stack([speeds_k, irms_arr, phase_arr])

# ── ARD 길이 스케일 (변수별 표준편차) ────────────────────────────────────────
LS_S = float(speeds_k.std())
LS_I = float(irms_arr.std())
LS_P = float(phase_arr.std())
print(f"  길이 스케일: ls_s={LS_S:.3f} kRPM | ls_I={LS_I:.1f} A | ls_P={LS_P:.2f} deg")

LAM = 1e-6

# ─────────────────────────────────────────────────────────────────────────────
# MODEL 1: 3D TPS RBF 모델 피팅 (106개 전체 센터)
# ─────────────────────────────────────────────────────────────────────────────
def _rbf_k_3d(s, irms, ph, s_c, i_c, p_c):
    r2 = (s - s_c)**2 / LS_S**2 + (irms - i_c)**2 / LS_I**2 + (ph - p_c)**2 / LS_P**2
    r = np.sqrt(r2)
    return r2 * np.log(r + 1e-12)

n = len(af_arr)
Phi_3d = np.zeros((n, n))
for j in range(n):
    Phi_3d[:, j] = _rbf_k_3d(speeds_k, irms_arr, phase_arr,
                             speeds_k[j], irms_arr[j], phase_arr[j])

rbf_weights_3d = np.linalg.solve(Phi_3d + LAM * np.eye(n), af_arr)

def af_from_rbf_3d(speed_rpm, irms_a, phase_deg):
    s   = np.asarray(speed_rpm, float) / 1000.0
    irm = np.asarray(irms_a,    float)
    ph  = np.asarray(phase_deg, float)
    s, irm, ph = np.broadcast_arrays(s, irm, ph)
    orig = s.shape
    sv, irmv, phv = s.ravel()[:, None], irm.ravel()[:, None], ph.ravel()[:, None]
    
    r2 = (sv - speeds_k)**2 / LS_S**2 + (irmv - irms_arr)**2 / LS_I**2 + (phv - phase_arr)**2 / LS_P**2
    r = np.sqrt(r2)
    K = r2 * np.log(r + 1e-12)
    result = K @ rbf_weights_3d
    return result.reshape(orig) if orig else float(result[0])

# ─────────────────────────────────────────────────────────────────────────────
# MODEL 2: 1D x 2D Separable RBF 모델 피팅
# ─────────────────────────────────────────────────────────────────────────────
base_idx = np.where(np.abs(speeds_k - 2.0) < 0.1)[0]
speeds_k_base = speeds_k[base_idx]
irms_arr_base = irms_arr[base_idx]
phase_arr_base = phase_arr[base_idx]
af_arr_base = af_arr[base_idx]

def _rbf_2d_k(irms, ph, i_c, p_c):
    r2 = (irms - i_c)**2 / LS_I**2 + (ph - p_c)**2 / LS_P**2
    r = np.sqrt(r2)
    return r2 * np.log(r + 1e-12)

n_base = len(base_idx)
Phi_g = np.zeros((n_base, n_base))
for j in range(n_base):
    Phi_g[:, j] = _rbf_2d_k(irms_arr_base, phase_arr_base,
                            irms_arr_base[j], phase_arr_base[j])

w_g = np.linalg.solve(Phi_g + LAM * np.eye(n_base), af_arr_base)

def predict_g(I, theta):
    I = np.asarray(I, float)
    theta = np.asarray(theta, float)
    I, theta = np.broadcast_arrays(I, theta)
    orig = I.shape
    Iv, thv = I.ravel()[:, None], theta.ravel()[:, None]
    
    r2 = (Iv - irms_arr_base)**2 / LS_I**2 + (thv - phase_arr_base)**2 / LS_P**2
    r = np.sqrt(r2)
    K = r2 * np.log(r + 1e-12)
    result = K @ w_g
    return result.reshape(orig) if orig else float(result[0])

# 1D 속도 배율 f(speed) 구하기 (4k, 8k, 16k RPM의 속도별 4점 사용)
other_speeds = [4.0, 8.0, 16.0]
target_currents = [115.0, 230.0, 345.0, 460.0]
selected_other_idx = []
for spd in other_speeds:
    spd_idx = np.where(np.abs(speeds_k - spd) < 0.1)[0]
    for i_val in target_currents:
        diffs = (irms_arr[spd_idx] - i_val)**2
        best_idx = spd_idx[np.argmin(diffs)]
        selected_other_idx.append(best_idx)
selected_other_idx = np.unique(selected_other_idx)

f_vals = []
for idx in selected_other_idx:
    spd = speeds_k[idx]
    I_val = irms_arr[idx]
    th_val = phase_arr[idx]
    af_actual = af_arr[idx]
    g_val = predict_g(I_val, th_val)
    f_val = af_actual / (g_val + 1e-12)
    f_vals.append((spd, f_val))

f_by_speed = {2.0: [1.0]}
for spd, f_val in f_vals:
    if spd not in f_by_speed:
        f_by_speed[spd] = []
    f_by_speed[spd].append(f_val)

speed_coords = []
f_coords = []
for spd in sorted(f_by_speed.keys()):
    speed_coords.append(spd)
    f_coords.append(np.mean(f_by_speed[spd]))

p_coeffs = np.polyfit(speed_coords, f_coords, 2)
p_func = np.poly1d(p_coeffs)

def af_from_rbf_separable(speed_rpm, irms_a, phase_deg):
    s = np.asarray(speed_rpm, float) / 1000.0
    irm = np.asarray(irms_a, float)
    ph = np.asarray(phase_deg, float)
    s, irm, ph = np.broadcast_arrays(s, irm, ph)
    orig = s.shape
    sv, irmv, phv = s.ravel(), irm.ravel(), ph.ravel()
    
    g_vals = predict_g(irmv, phv)
    f_vals = p_func(sv)
    result = f_vals * g_vals
    return result.reshape(orig) if orig else float(result[0])

# ── 기본 보정 함수 설정 (Separable 방식을 기본으로 사용) ──────────────────
def af_from_rbf(speed_rpm, irms_a, phase_deg):
    return af_from_rbf_separable(speed_rpm, irms_a, phase_deg)

# 다운스트림 호환용
def af_from_poly3d(speed_rpm, id_peak_a, iq_peak_a):
    idv = np.asarray(id_peak_a, float)
    iqv = np.asarray(iq_peak_a, float)
    irms  = np.sqrt(idv**2 + iqv**2) / np.sqrt(2)
    phase = np.degrees(np.arctan2(iqv, idv)) - 90.0
    return af_from_rbf(speed_rpm, irms, phase)

print("  af_from_rbf_3d() 및 af_from_rbf_separable() 수립 완료 (Separable 기본 활성화)")

# ── 3. Motor-CAD Lab 수식 포맷 ───────────────────────────────────────────────
# (1) 3D RBF 식 (106개 센터)
terms_3d = []
for j in range(n):
    w, s_c, i_c, p_c = rbf_weights_3d[j], speeds_k[j], irms_arr[j], phase_arr[j]
    r2_expr = f"((Speed/1000-{s_c:.4f})**2/{LS_S**2:.4f}+(Stator_Current_Phase_RMS-{i_c:.4f})**2/{LS_I**2:.4f}+(Phase_Advance-{p_c:.4f})**2/{LS_P**2:.4f})"
    term = f"({w:+.6f})*({r2_expr})*log({r2_expr}**0.5+1e-12)"
    terms_3d.append(term)
rbf_formula_3d = "Stator_Copper_Loss_AC * (\n  " + "\n  + ".join(terms_3d) + "\n) - Stator_Copper_Loss_AC"

# (2) Separable 식 (30개 센터)
terms_g = []
for j in range(n_base):
    w, i_c, p_c = w_g[j], irms_arr_base[j], phase_arr_base[j]
    r2_expr = f"((Stator_Current_Phase_RMS-{i_c:.4f})**2/{LS_I**2:.4f}+(Phase_Advance-{p_c:.4f})**2/{LS_P**2:.4f})"
    term = f"({w:+.6f})*({r2_expr})*log({r2_expr}**0.5+1e-12)"
    terms_g.append(term)
g_expr = " + ".join(terms_g)
f_expr = f"({p_coeffs[0]:+.6f}*(Speed/1000)**2{p_coeffs[1]:+.6f}*(Speed/1000){p_coeffs[2]:+.6f})"
rbf_formula_separable = "Stator_Copper_Loss_AC * (\n  " + f"({f_expr}) * (\n    {g_expr}\n  )" + "\n) - Stator_Copper_Loss_AC"

rbf_formula = rbf_formula_separable


  길이 스케일: ls_s=5.702 kRPM | ls_I=159.3 A | ls_P=31.39 deg
  af_from_rbf_3d() 및 af_from_rbf_separable() 수립 완료 (Separable 기본 활성화)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [6] 방법 A: AF(speed) 속도만 2차 다항식 + AF vs Speed 시각화
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

# ── 방법 A 피팅: 최대 전류에서 속도만의 2차 다항식 ──────────────────────────
max_curr     = curr_arr.max()
mask_maxcurr = np.isclose(curr_arr, max_curr, rtol=0.01)
spd_mc, af_mc = speeds_k[mask_maxcurr], af_arr[mask_maxcurr]

sort_idx = np.argsort(spd_mc)
spd_mc, af_mc = spd_mc[sort_idx], af_mc[sort_idx]

coeffs_A = np.polyfit(spd_mc, af_mc, deg=2)
af_A_fit = np.polyval(coeffs_A, spd_mc)
a2, a1, a0 = coeffs_A
_coeff_A = coeffs_A.copy()

def af_from_speed_only(speed_rpm):
    return np.polyval(_coeff_A, np.asarray(speed_rpm, float) / 1000.0)

lab_formula_extra = (
    f"(({a2:.6f}*(Speed/1000)^2 + {a1:.6f}*(Speed/1000) + {a0:.6f}) - 1)"
    f" * Stator_Copper_Loss_AC"
)

print("=== 방법 A: 속도만의 2차 다항식 (최대 전류 기준) ===")
print(f"  I_rms = {max_curr:.1f} A 기준")
print(f"  AF(s) = {a2:.6f}\u00b7s\u00b2 + {a1:.6f}\u00b7s + {a0:.6f}   (s: kRPM)\n")
for s, ref, fit in zip(spd_mc, af_mc, af_A_fit):
    print(f"    {s:.0f} kRPM: AF_ref={ref:.3f}, AF_fit={fit:.3f}, \u0394={fit-ref:+.3f}")
print(f"\n  [Motor-CAD Lab \uc218\uc2dd]\n  {lab_formula_extra}")

# ── AF vs Speed 시각화 ──────────────────────────────────────────────────────
unique_currents_s = sorted(set(round(p["current_rms"], 0) for p in af_points))
unique_phases_s   = sorted(set(round(p["phase_deg"],   0) for p in af_points))
unique_speeds_s   = sorted(set(p["speed_rpm"] for p in af_points))

n_curr_s  = len(unique_currents_s)
colors_s  = [plt.cm.plasma(i / max(1, n_curr_s - 1)) for i in range(n_curr_s)]
lstyles_s = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 5))]

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_title("Adjustment Factor  AF = FullFEA_AC / Hybrid_AC  vs Speed (\uc6b4\uc804\uc810\ubcc4)",
             fontsize=12, fontweight='bold')

for ki, curr in enumerate(unique_currents_s):
    for li, ph in enumerate(unique_phases_s):
        pts = sorted(
            [p for p in af_points
             if np.isclose(p["current_rms"], curr, atol=0.6)
             and np.isclose(p["phase_deg"],  ph,   atol=0.6)],
            key=lambda x: x["speed_rpm"]
        )
        if len(pts) < 2:
            continue
        spds = [p["speed_rpm"] for p in pts]
        afs  = [p["AF"]        for p in pts]
        ax.plot(spds, afs,
                marker='o', markersize=5,
                linestyle=lstyles_s[li % len(lstyles_s)],
                color=colors_s[ki], linewidth=1.5,
                label=f"I={curr:.0f} A, \u03c6={ph:.0f}\u00b0")

spd_fit  = np.linspace(min(unique_speeds_s) * 0.9, max(unique_speeds_s) * 1.05, 300)
af_fit_A = af_from_speed_only(spd_fit)
eq_str = f"y = {a2:.4f}\u00b7x\u00b2 {a1:+.4f}\u00b7x {a0:+.4f}  (x: kRPM)"
ax.plot(spd_fit, af_fit_A, 'k--', linewidth=2.5,
        label=f"Poly-A fit (I_max={max_curr:.0f} A)")
ax.text(0.97, 0.97, eq_str, transform=ax.transAxes, fontsize=9,
        va='top', ha='right', bbox=dict(boxstyle='round', fc='white', alpha=0.85))

ax.axhline(y=1.0, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label="AF = 1")
ax.set_xlabel("Speed [RPM]", fontsize=11)
ax.set_ylabel("Adjustment factor [-]", fontsize=11)
ax.legend(fontsize=7.5, loc='upper right', ncol=2, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("map_exports/AF_vs_speed_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("\uc800\uc7a5: map_exports/AF_vs_speed_curves.png")


# [6.5] 방법 B 시각화: id-iq 평면 AF 분포 (분리형 RBF)

속도별 id-iq 평면에서의 AF 예측 거동 및 분리형 RBF 곡면을 시각화합니다.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [6.5] 방법 B 시각화: id-iq 평면 AF 맵 (속도별)
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

unique_speeds_v = sorted(set(p["speed_rpm"] for p in af_points))
n_spd_v = len(unique_speeds_v)

fig, axes = plt.subplots(1, n_spd_v, figsize=(5.2 * n_spd_v, 4.8))
if n_spd_v == 1:
    axes = [axes]
fig.suptitle("Adjustment Factor  AF = FullFEA_AC / Hybrid_AC  (id-iq 평면)",
             fontsize=13, fontweight='bold')

af_vals_all = np.array([p["AF"] for p in af_points])
vmin_af = max(0.5, af_vals_all.min() - 0.1)
vmax_af = af_vals_all.max() + 0.1

for ax, spd in zip(axes, unique_speeds_v):
    pts  = [p for p in af_points if p["speed_rpm"] == spd]
    id_v = np.array([p["id_A"] for p in pts])
    iq_v = np.array([p["iq_A"] for p in pts])
    af_v = np.array([p["AF"]   for p in pts])

    sc = ax.scatter(id_v, iq_v, c=af_v, cmap='plasma', s=90,
                    edgecolors='k', linewidths=0.6,
                    vmin=vmin_af, vmax=vmax_af, zorder=3)
    for x, y, a in zip(id_v, iq_v, af_v):
        ax.annotate(f"{a:.2f}", (x, y), textcoords="offset points",
                    xytext=(5, 4), fontsize=7.5, color='black')

    pad = 80
    id_g = np.linspace(id_v.min() - pad, id_v.max() + pad, 50)
    iq_g = np.linspace(max(0, iq_v.min() - pad), iq_v.max() + pad, 50)
    ID, IQ = np.meshgrid(id_g, iq_g)
    AF_fit = af_from_poly3d(spd, ID.ravel(), IQ.ravel()).reshape(ID.shape)
    ct = ax.contour(ID, IQ, AF_fit, levels=8, cmap='coolwarm', alpha=0.65, linewidths=0.9)
    ax.clabel(ct, fmt="%.2f", fontsize=7.5)

    plt.colorbar(sc, ax=ax, label="AF [-]", shrink=0.85)
    ax.set_xlabel("$I_d$ [A, peak]", fontsize=9)
    ax.set_ylabel("$I_q$ [A, peak]", fontsize=9)
    ax.set_title(f"{spd/1000:.0f} kRPM", fontsize=11, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("map_exports/AF_map_visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: map_exports/AF_map_visualization.png")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [6.6] 방법 B 3D 표면 시각화: AF(id, iq) 곡면 (속도별)
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

unique_speeds_v = sorted(set(p["speed_rpm"] for p in af_points))
n_spd_v = len(unique_speeds_v)

fig = plt.figure(figsize=(5.5 * n_spd_v, 5.0))
fig.suptitle("AF Surface: AF(Id, Iq) 방법 B 3D 곡면 (속도별)", fontsize=13, fontweight="bold")

for k, spd in enumerate(unique_speeds_v):
    ax = fig.add_subplot(1, n_spd_v, k + 1, projection="3d")

    pts  = [p for p in af_points if p["speed_rpm"] == spd]
    id_v = np.array([p["id_A"] for p in pts])
    iq_v = np.array([p["iq_A"] for p in pts])
    af_v = np.array([p["AF"]   for p in pts])

    pad = 80
    id_g = np.linspace(id_v.min() - pad, id_v.max() + pad, 50)
    iq_g = np.linspace(max(0, iq_v.min() - pad), iq_v.max() + pad, 50)
    ID, IQ = np.meshgrid(id_g, iq_g)
    AF_fit = af_from_poly3d(spd, ID, IQ)

    surf = ax.plot_surface(ID, IQ, AF_fit, cmap="plasma", alpha=0.75,
                           linewidth=0, antialiased=True)
    ax.scatter(id_v, iq_v, af_v, c="red", s=60,
               edgecolors="k", linewidths=0.6, zorder=5, label="FEA data")

    fig.colorbar(surf, ax=ax, shrink=0.55, label="AF [-]")
    ax.set_xlabel("Id [A]", fontsize=8)
    ax.set_ylabel("Iq [A]", fontsize=8)
    ax.set_zlabel("AF [-]", fontsize=8)
    ax.set_title(f"{spd/1000:.0f} kRPM", fontsize=11, fontweight="bold")
    ax.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.savefig("map_exports/AF_3D_surface.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장 완료: map_exports/AF_3D_surface.png")


# [7] 대리 모델 성능 비교 및 시각화

- 3D TPS RBF 모델과 1D x 2D Separable RBF 모델의 예측 오차(Train MAE 및 Leave-One-Out CV 오차)를 직접 비교합니다.
- 두 모델의 예측 데이터 Parity Plot 및 3-way Boxplot을 생성하여 비교 시각화하고 최종 JSON 데이터를 내보냅니다.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [7] RBF 모델 비교 검증: 3D TPS RBF vs. 1D x 2D Separable RBF vs. FullFEA
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

print(f"=== RBF 보정 오차 검증 및 비교: 3D RBF vs Separable vs FullFEA ===\n")

# ── 데이터 배열 빌드 (이름 충돌 방지를 위해 접미사 _arr 사용)
h_ac_arr = np.array([p["hybrid_ac_kW"] for p in af_points])
f_ac_arr = np.array([p["fea_ac_kW"] for p in af_points])

# ── 훈련 세트 오차 계산
err_raw, err_3d, err_sep = [], [], []
rows = []
for p in af_points:
    spd   = p["speed_rpm"]
    irms  = p["current_rms"]
    phase = p["phase_deg"]
    h_ac  = p["hybrid_ac_kW"]
    f_ac  = p["fea_ac_kW"]
    
    af_3d  = float(af_from_rbf_3d(spd, irms, phase))
    af_sep = float(af_from_rbf_separable(spd, irms, phase))
    
    corr_3d  = h_ac * af_3d
    corr_sep = h_ac * af_sep
    
    e_raw = (h_ac - f_ac) / (f_ac + 1e-12) * 100
    e_3d  = (corr_3d - f_ac) / (f_ac + 1e-12) * 100
    e_sep = (corr_sep - f_ac) / (f_ac + 1e-12) * 100
    
    err_raw.append(e_raw)
    err_3d.append(e_3d)
    err_sep.append(e_sep)
    rows.append((spd, irms, phase, h_ac, f_ac, corr_3d, corr_sep, e_raw, e_3d, e_sep))

ea = np.array(err_raw)
e3 = np.array(err_3d)
es = np.array(err_sep)

# ── LOOCV (Leave-One-Out Cross Validation) 계산 ───────────────────────────────
print("  LOOCV 계산 중 (약 1.5초 소요)... ")

# (1) 3D RBF LOOCV
loocv_errors_3d = []
for i in range(n):
    X_tr = np.delete(X_data, i, axis=0)
    y_tr = np.delete(af_arr, i, axis=0)
    Phi_tr = np.zeros((n-1, n-1))
    for j in range(n-1):
        Phi_tr[:, j] = _rbf_k_3d(X_tr[:, 0], X_tr[:, 1], X_tr[:, 2],
                                 X_tr[j, 0], X_tr[j, 1], X_tr[j, 2])
    w_tr = np.linalg.solve(Phi_tr + LAM * np.eye(n-1), y_tr)
    
    r2 = (X_data[i, 0] - X_tr[:, 0])**2 / LS_S**2 \
       + (X_data[i, 1] - X_tr[:, 1])**2 / LS_I**2 \
       + (X_data[i, 2] - X_tr[:, 2])**2 / LS_P**2
    r = np.sqrt(r2)
    K = r2 * np.log(r + 1e-12)
    y_pred = K @ w_tr
    corr_val = h_ac_arr[i] * y_pred
    loocv_errors_3d.append(abs((corr_val - f_ac_arr[i]) / f_ac_arr[i] * 100))
mae_loocv_3d = np.mean(loocv_errors_3d)

# (2) Separable RBF LOOCV
loocv_errors_sep = []
for i in range(n):
    base_train_idx = [idx for idx in base_idx if idx != i]
    X_base_tr = X_data[base_train_idx, 1:3]
    y_base_tr = af_arr[base_train_idx]
    
    Phi_g_tr = np.zeros((len(base_train_idx), len(base_train_idx)))
    for j in range(len(base_train_idx)):
        Phi_g_tr[:, j] = _rbf_2d_k(X_base_tr[:, 0], X_base_tr[:, 1],
                                    X_base_tr[j, 0], X_base_tr[j, 1])
    w_g_tr = np.linalg.solve(Phi_g_tr + LAM * np.eye(len(base_train_idx)), y_base_tr)
    
    def predict_g_tr(I, theta):
        I = np.asarray(I, float)
        theta = np.asarray(theta, float)
        I, theta = np.broadcast_arrays(I, theta)
        orig = I.shape
        Iv, thv = I.ravel()[:, None], theta.ravel()[:, None]
        r2 = (Iv - X_base_tr[:, 0])**2 / LS_I**2 + (thv - X_base_tr[:, 1])**2 / LS_P**2
        r = np.sqrt(r2)
        K = r2 * np.log(r + 1e-12)
        result = K @ w_g_tr
        return result.reshape(orig) if orig else float(result[0])
        
    cal_train_idx = [idx for idx in selected_other_idx if idx != i]
    f_vals_tr = []
    for idx in cal_train_idx:
        spd = speeds_k[idx]
        I_val = irms_arr[idx]
        th_val = phase_arr[idx]
        af_actual = af_arr[idx]
        g_val = predict_g_tr(I_val, th_val)
        f_val = af_actual / (g_val + 1e-12)
        f_vals_tr.append((spd, f_val))
        
    f_by_speed_tr = {2.0: [1.0]}
    for spd, f_val in f_vals_tr:
        if spd not in f_by_speed_tr:
            f_by_speed_tr[spd] = []
        f_by_speed_tr[spd].append(f_val)
        
    speed_coords_tr = []
    f_coords_tr = []
    for spd in sorted(f_by_speed_tr.keys()):
        speed_coords_tr.append(spd)
        f_coords_tr.append(np.mean(f_by_speed_tr[spd]))
        
    p_coeffs_tr = np.polyfit(speed_coords_tr, f_coords_tr, 2)
    p_func_tr = np.poly1d(p_coeffs_tr)
    
    g_val_i = predict_g_tr(X_data[i, 1], X_data[i, 2])
    f_val_i = p_func_tr(X_data[i, 0])
    y_pred_i = f_val_i * g_val_i
    corr_val = h_ac_arr[i] * y_pred_i
    loocv_errors_sep.append(abs((corr_val - f_ac_arr[i]) / f_ac_arr[i] * 100))
mae_loocv_sep = np.mean(loocv_errors_sep)

print("=== RBF 보정 오차 최종 비교 결과 ===")
print(f"  1) Hybrid (보정 전):        Train MAE={np.abs(ea).mean():.2f}% | MaxAE={np.abs(ea).max():.2f}%")
print(f"  2) 3D TPS RBF:             Train MAE={np.abs(e3).mean():.2f}% | MaxAE={np.abs(e3).max():.2f}% | LOOCV MAE={mae_loocv_3d:.2f}%")
print(f"  3) Separable (분리형 RBF):   Train MAE={np.abs(es).mean():.2f}% | MaxAE={np.abs(es).max():.2f}% | LOOCV MAE={mae_loocv_sep:.2f}%")

# ── 시각화 및 그림 저장
fea_all   = np.array([r[4] for r in rows])
hybr_all  = np.array([r[3] for r in rows])
corr_3d   = np.array([r[5] for r in rows])
corr_sep  = np.array([r[6] for r in rows])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("RBF Model Comparison: 3D RBF vs Separable RBF vs FullFEA", fontsize=12, fontweight='bold')

# [왼쪽] Parity Plot
ax = axes[0]
lim = [min(fea_all.min(), hybr_all.min(), corr_3d.min(), corr_sep.min()) * 0.9,
       max(fea_all.max(), hybr_all.max(), corr_3d.max(), corr_sep.max()) * 1.05]
ax.plot(lim, lim, 'k--', linewidth=1.2, label='Perfect fit')
ax.scatter(fea_all, hybr_all, c='grey',      s=30, alpha=0.5, label='Hybrid (보정 전)', zorder=2)
ax.scatter(fea_all, corr_3d,  c='steelblue', s=45, alpha=0.7, label=f'3D RBF (LOOCV: {mae_loocv_3d:.2f}%)', zorder=3)
ax.scatter(fea_all, corr_sep, c='tomato',    s=45, alpha=0.8, label=f'Separable (LOOCV: {mae_loocv_sep:.2f}%)', zorder=4)
ax.set_xlabel("FullFEA AC Loss [kW]", fontsize=10)
ax.set_ylabel("Predicted AC Loss [kW]", fontsize=10)
ax.set_title("Parity Plot", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlim(lim); ax.set_ylim(lim)

# [오른쪽] Boxplot
ax2 = axes[1]
bp = ax2.boxplot([ea, e3, es], labels=['Hybrid (보정 전)', '3D RBF', 'Separable RBF'],
                 patch_artist=True, widths=0.4)
bp['boxes'][0].set_facecolor('grey');       bp['boxes'][0].set_alpha(0.4)
bp['boxes'][1].set_facecolor('steelblue');  bp['boxes'][1].set_alpha(0.6)
bp['boxes'][2].set_facecolor('tomato');     bp['boxes'][2].set_alpha(0.6)
ax2.axhline(0, color='k', linestyle='--', linewidth=1)
ax2.set_ylabel("오차 [%]", fontsize=10)
ax2.set_title("Error Distribution Comparison", fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.4)
for i, (arr, x) in enumerate([(ea, 1), (e3, 2), (es, 3)]):
    ax2.text(x, arr.max() + 0.5, f"MAE={np.abs(arr).mean():.1f}%",
             ha='center', fontsize=8.5, color='black')

plt.tight_layout()
plt.savefig("map_exports/RBF_correction_validation.png", dpi=150, bbox_inches='tight')
plt.show()
print("그림 저장 완료: map_exports/RBF_correction_validation.png")

# ── 모델 내보내기 및 저장
export = {
    "model_type": "RBF_Comparison",
    "3D_model": {
        "model": "3D_TPS_RBF",
        "n_centers": int(n),
        "weights": rbf_weights_3d.tolist(),
        "validation": {
            "Train_MAE_pct": float(np.abs(e3).mean()),
            "LOOCV_MAE_pct": float(mae_loocv_3d),
        },
        "mcad_formula": rbf_formula_3d
    },
    "separable_model": {
        "model": "Separable_1D_2D_RBF",
        "n_base_centers": int(n_base),
        "base_weights": w_g.tolist(),
        "speed_poly_coeffs": p_coeffs.tolist(),
        "validation": {
            "Train_MAE_pct": float(np.abs(es).mean()),
            "LOOCV_MAE_pct": float(mae_loocv_sep),
        },
        "mcad_formula": rbf_formula_separable
    },
    "mcad_formula_full": rbf_formula_3d,
    "mcad_formula_reduced_30": rbf_formula_separable,
    "mcad_formula_top20": rbf_formula_separable,
    "length_scales": {"LS_S_kRPM": float(LS_S), "LS_I_A": float(LS_I), "LS_P_deg": float(LS_P)},
    "af_points": af_points
}
save_path = Path("map_exports/AF_RBF_model.json")
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print(f"\n  JSON 모델 저장 완료: {save_path}")

# ── [대화형 4-Way 비교 플롯 구현] ───────────────────────────────────────────
try:
    import IPython
    shell = IPython.get_ipython()
    if shell is not None:
        import os, sys
        has_vscode_env = any(k.startswith('VSCODE_') for k in os.environ.keys())
        has_vscode_modules = any('vscode' in m.lower() for m in sys.modules.keys())
        selected_backend = 'widget' if (has_vscode_env and has_vscode_modules) else 'inline'
        if selected_backend == 'widget':
            shell.run_line_magic('matplotlib', 'widget')
        else:
            shell.run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

print("\n  대화형 4-Way 비교 플롯 로딩...")
speeds = np.array([p["speed_rpm"] for p in af_points])
irms = np.array([p["current_rms"] for p in af_points])
phases = np.array([p["phase_deg"] for p in af_points])
id_vals = np.array([p["id_A"] for p in af_points])
iq_vals = np.array([p["iq_A"] for p in af_points])

loss_hyb = np.array([p["hybrid_ac_kW"] for p in af_points])
loss_fea = np.array([p["fea_ac_kW"] for p in af_points])
loss_3d  = np.array([float(af_from_rbf_3d(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in af_points])
loss_sep = np.array([float(af_from_rbf_separable(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in af_points])

fig_int = plt.figure(figsize=(17, 8.5))
fig_int.suptitle("AC Loss 3D Map & Speed Curve Comparison: Hybrid vs 3D RBF vs Separable RBF vs FullFEA", fontsize=13, fontweight='bold')

ax_hyb = fig_int.add_subplot(2, 3, 1, projection='3d')
ax_3d  = fig_int.add_subplot(2, 3, 2, projection='3d')
ax_sep = fig_int.add_subplot(2, 3, 4, projection='3d')
ax_fea = fig_int.add_subplot(2, 3, 5, projection='3d')
ax_curve = fig_int.add_subplot(2, 3, (3, 6))

ax_hyb.set_title("1) Hybrid (보정 전)", fontsize=11, fontweight='bold')
ax_3d.set_title("2) 3D TPS RBF (보정 후)", fontsize=11, fontweight='bold')
ax_sep.set_title("3) Separable RBF (보정 후)", fontsize=11, fontweight='bold')
ax_fea.set_title("4) FullFEA (참조값)", fontsize=11, fontweight='bold')

unique_speeds = sorted(list(set(speeds)))
speed_colors = {2000: 'cyan', 4000: 'limegreen', 8000: 'orange', 16000: 'tomato'}
default_colors = ['cyan', 'limegreen', 'orange', 'tomato']
axes_3d = [ax_hyb, ax_3d, ax_sep, ax_fea]
losses_list = [loss_hyb, loss_3d, loss_sep, loss_fea]

legend_patches = []
for i, spd in enumerate(unique_speeds):
    color = speed_colors.get(spd, default_colors[i % len(default_colors)])
    legend_patches.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
    idx_spd = (speeds == spd)
    if np.any(idx_spd) and np.sum(idx_spd) >= 3:
        for ax, loss_val in zip(axes_3d, losses_list):
            ax.plot_trisurf(id_vals[idx_spd], iq_vals[idx_spd], loss_val[idx_spd], color=color, edgecolor='none', alpha=0.2)

sc_hyb = ax_hyb.scatter(id_vals, iq_vals, loss_hyb, c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
sc_3d  = ax_3d.scatter(id_vals, iq_vals, loss_3d,   c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
sc_sep = ax_sep.scatter(id_vals, iq_vals, loss_sep, c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
sc_fea = ax_fea.scatter(id_vals, iq_vals, loss_fea, c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
scatters = [sc_hyb, sc_3d, sc_sep, sc_fea]

for ax in axes_3d:
    ax.set_xlabel("I_d [A]", fontsize=8, labelpad=5)
    ax.set_ylabel("I_q [A]", fontsize=8, labelpad=5)
    ax.set_zlabel("AC Loss [kW]", fontsize=8, labelpad=5)
    ax.legend(handles=legend_patches, fontsize=8, loc="upper right")

ax_curve.text(0.5, 0.5, "3D 플롯에서 임의의 점을 클릭한 후\nSpacebar를 누르거나 클릭하면 우측에 속도별 비교 곡선이 출력됩니다.", 
             ha="center", va="center", fontsize=10, color="gray")
ax_curve.set_xlabel("Speed [RPM]", fontsize=9)
ax_curve.set_ylabel("AC Loss [kW]", fontsize=9)
ax_curve.grid(True, linestyle="--", alpha=0.5)

selected_pt = {"current_rms": None, "phase_deg": None, "id_A": None, "iq_A": None}
highlights = []
annots = []
for ax in axes_3d:
    annot = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
    annot.set_visible(False)
    annots.append(annot)

def update_2d_curve(curr, ph):
    ax_curve.clear()
    match_pts = [p for p in af_points if np.isclose(p["current_rms"], curr) and np.isclose(p["phase_deg"], ph)]
    match_pts = sorted(match_pts, key=lambda x: x["speed_rpm"])
    
    curve_speeds = [p["speed_rpm"] for p in match_pts]
    c_loss_hyb = [p["hybrid_ac_kW"] for p in match_pts]
    c_loss_fea = [p["fea_ac_kW"] for p in match_pts]
    c_loss_3d  = [float(af_from_rbf_3d(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in match_pts]
    c_loss_sep = [float(af_from_rbf_separable(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in match_pts]
    
    ax_curve.plot(curve_speeds, c_loss_hyb, marker='o', linestyle='-',  color='grey',      linewidth=1.5, label="1) Hybrid (보정 전)")
    ax_curve.plot(curve_speeds, c_loss_3d,  marker='s', linestyle='-',  color='steelblue', linewidth=2,   label="2) 3D RBF (보정 후)")
    ax_curve.plot(curve_speeds, c_loss_sep, marker='^', linestyle='-',  color='tomato',    linewidth=2,   label="3) Separable (보정 후)")
    ax_curve.plot(curve_speeds, c_loss_fea, marker='*', linestyle='--', color='black',     linewidth=2,   label="4) FullFEA Reference")
    
    for xs, yh, y3, ys, yf in zip(curve_speeds, c_loss_hyb, c_loss_3d, c_loss_sep, c_loss_fea):
        ax_curve.annotate(f"{yh:.2f}", xy=(xs, yh), xytext=(4, 8),   textcoords="offset points", fontsize=8, color="grey")
        ax_curve.annotate(f"{y3:.2f}", xy=(xs, y3), xytext=(4, 0),   textcoords="offset points", fontsize=8, color="steelblue")
        ax_curve.annotate(f"{ys:.2f}", xy=(xs, ys), xytext=(4, -8),  textcoords="offset points", fontsize=8, color="tomato")
        ax_curve.annotate(f"{yf:.2f}", xy=(xs, yf), xytext=(4, -16), textcoords="offset points", fontsize=8, color="black")
        
    ax_curve.set_title(f"AC Loss vs Speed Comparison\n(I_rms={curr:.1f}A, Phase={ph:.1f}°)", fontsize=11, fontweight='bold')
    ax_curve.set_xlabel("Speed [RPM]", fontsize=9)
    ax_curve.set_ylabel("AC Loss [kW]", fontsize=9)
    ax_curve.grid(True, linestyle="--", alpha=0.5)
    ax_curve.legend(fontsize=9, loc="upper left")

def on_pick(event):
    if event.artist not in scatters:
        return
    idx = event.ind[0]
    p_sel = af_points[idx]
    curr = p_sel["current_rms"]
    ph = p_sel["phase_deg"]
    
    selected_pt["current_rms"] = curr
    selected_pt["phase_deg"] = ph
    selected_pt["id_A"] = p_sel["id_A"]
    selected_pt["iq_A"] = p_sel["iq_A"]
    
    for h in highlights:
        h.remove()
    highlights.clear()
    
    same_pt_idx = np.where((irms == curr) & (phases == ph))[0]
    for ax, loss_val in zip(axes_3d, losses_list):
        h = ax.scatter(id_vals[same_pt_idx], iq_vals[same_pt_idx], loss_val[same_pt_idx], color='red', s=60, edgecolors='black', linewidths=1.5, zorder=10)
        highlights.append(h)
        
    msg = f"Selected: I_rms={curr:.1f}A, Phase={ph:.1f}°\nId={selected_pt['id_A']:.1f}A, Iq={selected_pt['iq_A']:.1f}A"
    for annot in annots:
        annot.set_text(msg)
        annot.set_visible(True)
        
    update_2d_curve(curr, ph)
    fig_int.canvas.draw_idle()

def on_key(event):
    if event.key != ' ' or selected_pt["current_rms"] is None:
        return
    update_2d_curve(selected_pt["current_rms"], selected_pt["phase_deg"])
    fig_int.canvas.draw_idle()

fig_int.canvas.mpl_connect('pick_event', on_pick)
fig_int.canvas.mpl_connect('key_press_event', on_key)
plt.tight_layout()
plt.show()


=== RBF 보정 오차 검증 및 비교: 3D RBF vs Separable vs FullFEA ===

  LOOCV 계산 중 (약 1.5초 소요)... 
=== RBF 보정 오차 최종 비교 결과 ===
  1) Hybrid (보정 전):        Train MAE=39.70% | MaxAE=78.23%
  2) 3D TPS RBF:             Train MAE=0.00% | MaxAE=0.00% | LOOCV MAE=4.95%
  3) Separable (분리형 RBF):   Train MAE=5.02% | MaxAE=40.69% | LOOCV MAE=8.49%
그림 저장 완료: map_exports/RBF_correction_validation.png

  JSON 모델 저장 완료: map_exports\AF_RBF_model.json

  대화형 4-Way 비교 플롯 로딩...


<string>:174: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from current font.
<string>:174: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from current font.
<string>:174: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
<string>:174: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from current font.
<string>:174: UserWarning: Glyph 52264 (\N{HANGUL SYLLABLE CA}) missing from current font.
<string>:175: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from current font.
<string>:175: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from current font.
<string>:175: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
<string>:175: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from current font.
<string>:175: UserWarning: Glyph 52264 (\N{HANGUL SYLLABLE CA}) missing from current font.
<string>:366: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from curre